
# Problem 4 - Iterative Policy Evaluation on the 5x5 gridworld (EE 5531 Assignment-2).

Reference implementation written to explain the algorithm.

Setup taken from Figure 1 of the assignment:
  - 5x5 grid, terminals at (0,0) and (4,4)
  - actions up/down/left/right, deterministic
  - moving off the grid leaves you where you are
  - r = -1 on every transition out of a non-terminal state
  - gamma = 0.95
  - equiprobable random policy, pi(a|s) = 1/4

In [3]:
import numpy as np

In [4]:
GAMMA = 0.95
REWARD = -1.0
TERMINALS = {(0, 0), (4, 4)}

In [5]:
# (row delta, col delta). Row 0 is the top row, so "up" decreases the row index.
ACTIONS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}

In [6]:
def step(row, col, action):
    
    dr, dc = ACTIONS[action]
    next_row, next_col = row + dr, col + dc
    if next_row < 0 or next_row > 4 or next_col < 0 or next_col > 4:
        return row, col
    return next_row, next_col

In [7]:
def policy_evaluation(gamma=GAMMA, theta=1e-8, max_sweeps=10000):
    
    values = np.zeros((5, 5))
    history = [values.copy()]
    deltas = []

    for sweep in range(1, max_sweeps + 1):
        new_values = np.zeros((5, 5))

        for row in range(5):
            for col in range(5):
                # Terminal states have value 0 by definition - no backup here.
                if (row, col) in TERMINALS:
                    continue

                # Average the one-step backup over the four equiprobable actions.
                total = 0.0
                for action in ACTIONS:
                    next_row, next_col = step(row, col, action)
                    total += 0.25 * (REWARD + gamma * values[next_row, next_col])
                new_values[row, col] = total

        delta = np.abs(new_values - values).max()
        values = new_values
        history.append(values.copy())
        deltas.append(delta)

        if delta < theta:
            break

    return values, history, deltas, sweep

In [8]:
def exact_solution(gamma=GAMMA):
    
    """Solve v_pi = r_pi + gamma * P_pi v_pi directly, as an independent check."""
    states = [(r, c) for r in range(5) for c in range(5)]
    index = {s: i for i, s in enumerate(states)}
    n = len(states)

    P = np.zeros((n, n))
    r = np.zeros(n)

    for s in states:
        i = index[s]
        if s in TERMINALS:
            continue  # row of zeros, r = 0 -> v = 0
        r[i] = REWARD
        for action in ACTIONS:
            j = index[step(s[0], s[1], action)]
            P[i, j] += 0.25

    v = np.linalg.solve(np.eye(n) - gamma * P, r)
    return v.reshape(5, 5)

In [9]:
def show(values, title):
    print(title)
    for row in range(5):
        print("  " + "  ".join(f"{values[row, col]:7.3f}" for col in range(5)))
    print()

In [10]:
if __name__ == "__main__":
    
    values, history, deltas, sweeps = policy_evaluation()

    print(f"gamma = {GAMMA}, reward = {REWARD}, terminals = {sorted(TERMINALS)}")
    print(f"converged after {sweeps} sweeps (theta = 1e-8)\n")

    show(history[1], "after sweep 1:")
    show(history[2], "after sweep 2:")
    show(history[3], "after sweep 3:")
    show(values, f"converged values (sweep {sweeps}):")

    exact = exact_solution()
    show(exact, "direct linear solve, for comparison:")
    print(f"max difference between the two: {np.abs(values - exact).max():.3e}\n")

    print("delta per sweep (first 12):")
    print("  " + ", ".join(f"{d:.4f}" for d in deltas[:12]))
    print(f"\nratio delta_k+1 / delta_k around sweep 20: "
          f"{deltas[20] / deltas[19]:.4f}   (gamma = {GAMMA})")

    # Sanity checks
    assert values[0, 0] == 0.0 and values[4, 4] == 0.0, "terminals must stay 0"
    assert values.shape == (5, 5)
    assert (values[np.array([[(r, c) not in TERMINALS for c in range(5)]
                             for r in range(5)])] < 0).all(), "non-terminals negative"
    print("\nchecks passed: shape 5x5, terminals 0, all other states negative")

    # The assignment calls the value "expected number of steps to termination".
    # That reading is only exact at gamma = 1; show both.
    undisc, _, _, n_undisc = policy_evaluation(gamma=1.0, theta=1e-10)
    show(-undisc, f"gamma = 1.0: expected number of steps ({n_undisc} sweeps)")
    print(f"most-discounted state value at gamma=0.95: {values.min():.3f}"
          f"   (floor is -1/(1-gamma) = {-1 / (1 - GAMMA):.1f})")

gamma = 0.95, reward = -1.0, terminals = [(0, 0), (4, 4)]
converged after 235 sweeps (theta = 1e-8)

after sweep 1:
    0.000   -1.000   -1.000   -1.000   -1.000
   -1.000   -1.000   -1.000   -1.000   -1.000
   -1.000   -1.000   -1.000   -1.000   -1.000
   -1.000   -1.000   -1.000   -1.000   -1.000
   -1.000   -1.000   -1.000   -1.000    0.000

after sweep 2:
    0.000   -1.713   -1.950   -1.950   -1.950
   -1.713   -1.950   -1.950   -1.950   -1.950
   -1.950   -1.950   -1.950   -1.950   -1.950
   -1.950   -1.950   -1.950   -1.950   -1.713
   -1.950   -1.950   -1.950   -1.712    0.000

after sweep 3:
    0.000   -2.333   -2.796   -2.853   -2.853
   -2.333   -2.740   -2.853   -2.853   -2.853
   -2.796   -2.853   -2.853   -2.853   -2.796
   -2.853   -2.853   -2.853   -2.740   -2.333
   -2.853   -2.853   -2.796   -2.333    0.000

converged values (sweep 235):
    0.000   -8.864  -12.686  -14.285  -14.829
   -8.864  -11.561  -13.371  -14.136  -14.285
  -12.686  -13.371  -13.702  -13.371  -